In [1]:
! pip install pillow

In [1]:
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import os
import random

base_dir = "data"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Polices
font_candidates = [
    ("DejaVuSans-Bold.ttf", 60),
    ("DejaVuSans.ttf", 60),
    ("DejaVuSerif-Bold.ttf", 60),
    ("DejaVuSansMono-Bold.ttf", 60),
]

fonts = []
for name, size in font_candidates:
    try:
        fonts.append(ImageFont.truetype(name, size))
    except:
        pass

if not fonts:
    fonts = [ImageFont.load_default()]


def add_light_noise(img, amount=50):
    pixels = img.load()
    w, h = img.size
    for _ in range(amount):
        x = random.randint(0, w - 1)
        y = random.randint(0, h - 1)
        pixels[x, y] = random.choice([0, 255])
    return img


def draw_centered_text(draw, text, font, size, dx=0, dy=0):
    bbox = draw.textbbox((0, 0), text, font=font)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]

    x = (size[0] - tw) // 2 + dx
    y = (size[1] - th) // 2 + dy

    draw.text((x, y), text, fill=0, font=font)


def create_digit_image(text, path, size=(120, 120), variant_type="clean"):
    img = Image.new("L", size, color=255)
    draw = ImageDraw.Draw(img)

    font = random.choice(fonts)

    # Décalage léger
    dx = random.randint(-3, 3)
    dy = random.randint(-3, 3)

    draw_centered_text(draw, text, font, size, dx=dx, dy=dy)

    # Variantes contrôlées
    if variant_type == "clean":
        if random.random() < 0.2:
            img = img.rotate(random.randint(-4, 4), fillcolor=255)

    elif variant_type == "rot":
        img = img.rotate(random.randint(-8, 8), fillcolor=255)

    elif variant_type == "noise":
        img = add_light_noise(img, amount=random.randint(25, 70))

    elif variant_type == "blur":
        img = img.filter(ImageFilter.GaussianBlur(radius=0.5))

    elif variant_type == "mixed":
        if random.random() < 0.5:
            img = img.rotate(random.randint(-6, 6), fillcolor=255)
        if random.random() < 0.5:
            img = add_light_noise(img, amount=random.randint(20, 60))
        if random.random() < 0.3:
            img = img.filter(ImageFilter.GaussianBlur(radius=0.5))

    img.save(path)


def create_test_image(code, path, size=(420, 120)):
    img = Image.new("L", size, color=255)
    draw = ImageDraw.Draw(img)

    font = random.choice(fonts)

    dx = random.randint(-8, 8)
    dy = random.randint(-4, 4)

    draw_centered_text(draw, code, font, size, dx=dx, dy=dy)

    # Test : léger réalisme, pas trop dur
    if random.random() < 0.4:
        img = img.rotate(random.randint(-5, 5), fillcolor=255)

    if random.random() < 0.2:
        img = img.filter(ImageFilter.GaussianBlur(radius=0.5))

    if random.random() < 0.25:
        img = add_light_noise(img, amount=random.randint(30, 80))

    img.save(path)


# Nettoyage ancien dataset
for folder in [train_dir, test_dir]:
    for f in os.listdir(folder):
        if f.lower().endswith(".png"):
            os.remove(os.path.join(folder, f))


# =========================
# TRAIN : 100 images
# 10 chiffres x 10 variantes
# =========================
variant_cycle = [
    "clean",
    "clean",
    "rot",
    "noise",
    "blur",
    "mixed",
    "clean",
    "rot",
    "noise",
    "mixed",
]

for digit in range(10):
    for i in range(1, 11):
        variant = variant_cycle[i - 1]
        filename = f"{digit}_{i}.png"
        path = os.path.join(train_dir, filename)
        create_digit_image(str(digit), path, variant_type=variant)


# =========================
# TEST : 15 images
# =========================
test_codes = [
    "12345", "59130", "62487", "75001", "33000",
    "69000", "13001", "06000", "21000", "44000",
    "34000", "67000", "31000", "80000", "29200"
]

for code in test_codes:
    create_test_image(code, os.path.join(test_dir, f"{code}.png"))

print("Dataset AMELIORE généré ✔")
print("Train :", len(os.listdir(train_dir)), "images")
print("Test  :", len(os.listdir(test_dir)), "images")

Dataset AMELIORE généré ✔
Train : 100 images
Test  : 15 images
